# Landing → Bronze pipeline

Loads the raw extracts from the landing volume (`coreview.landing.organizations_data`) into the bronze layer. Three sources, each following the same pattern — read, stamp the load time, overwrite the bronze table:

| Source | Format | Bronze output |
| --- | --- | --- |
| CRM accounts | CSV | `coreview.bronze.crm_accounts` |
| Organizations (dated snapshots) | JSONL | `coreview.bronze.organizations` |
| Sign-in events | CSV | `coreview.bronze.signins` |

Bronze keeps the data raw (all strings, no deduplication, no casting) and is fully refreshed on every run. Downstream: `bronze_to_silver` cleans and deduplicates into silver, `silver_to_gold` builds the gold layer.

In [0]:
# --- one-time setup: catalog, schemas and the landing volume ---
# IF NOT EXISTS makes the whole cell safe to re-run on every pipeline execution
spark.sql("CREATE CATALOG IF NOT EXISTS coreview")
spark.sql("CREATE SCHEMA IF NOT EXISTS coreview.landing")
spark.sql("CREATE SCHEMA IF NOT EXISTS coreview.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS coreview.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS coreview.gold")
spark.sql("CREATE VOLUME IF NOT EXISTS coreview.landing.organizations_data")

## Accounts
Load the CRM accounts extract from the landing volume into bronze. We want to capture every change for every account, so nothing is deduplicated at this layer - bronze keeps the raw extract as-is.

In [0]:
# --- imports (StructType is needed by the flatten helper further below) ---
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
# landing volume location of the CRM accounts extract (CSV files)
accounts_path = "/Volumes/workspace/default/organizations_data/data/accounts"

In [0]:
# read every CSV file in the folder as one DataFrame; columns stay raw
# strings - no parsing or casting happens in the bronze layer
accounts_df = spark.read.format('csv').option('header', 'true').load(accounts_path)

### Stamp the load time
Every bronze row carries the ingestion timestamp; its date part is also the partition key of the bronze table.

In [0]:
# Ingestion_date doubles as the partition column of the bronze table below
accounts_df = (accounts_df.withColumn("Ingestion_timestamp", F.current_timestamp())
                        .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

In [0]:
# --- debug preview of the raw accounts data (DISABLED) ---
# commented out to keep the pipeline output clean;
# uncomment while exploring if you need a peek at the data
# display(accounts_df)

In [0]:
# full refresh: bronze always mirrors the current landing extract
# (the incremental overwrite-then-append variant is kept, commented, below)
acc_table_name = "coreview.bronze.crm_accounts"
accounts_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(acc_table_name)

In [0]:
# quick sanity check on the loaded row count
print(f"accounts: {accounts_df.count()} rows")

### Write mode
Bronze tables are **overwritten** on every run - the landing extracts are complete snapshots, so a full refresh keeps bronze an exact mirror of the source. For incremental ingestion, switch to the commented variant below: overwrite only on the initial run, append afterwards.

In [0]:
# --- incremental write variant (DISABLED - kept for reference) ---
# appends when the bronze table already exists, overwrites only on the very
# first load. Switched off because the landing files are complete extracts
# and the active cell above does a full refresh each run, which is simpler
# and guarantees bronze matches the source exactly.

# acc_table_name = "coreview.bronze.crm_accounts"

# if spark.catalog.tableExists(acc_table_name):
#     accounts_df.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(acc_table_name)
# else:
#     accounts_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(acc_table_name)


In [0]:
# --- quick check of the loaded table (DISABLED) ---
# uncomment to preview the bronze table after the load
# %%sql
# select * from coreview.bronze.crm_accounts

## Organizations
The landing folder holds dated JSONL snapshots of every organization. We want the full change history, so nothing is deduplicated here - each file's state is preserved, and the file name (which encodes the state date) is kept as a column.

In [0]:
# landing volume location of the dated organization snapshots (JSONL files)
organisations_path = "/Volumes/workspace/default/organizations_data/data/organizations/"

### Why the file name is captured
Each JSONL file name contains a date - the date of the organization state stored in that file. That is why `_metadata.file_name` is kept as a column below: the silver layer later derives `capture_date` from it, which drives the whole organization history (SCD2) downstream.

In [0]:
# one JSON object per line (multiLine=false); the source file name is kept
# as a column because it encodes the snapshot date (see the note above)
org_df = (
    spark.read
    .option("multiLine", "false")
    .json(organisations_path + "*.jsonl")
    .withColumn("source_file_name", F.col("_metadata.file_name"))
)

In [0]:
# inspect the raw, nested schema - this nesting is why the flatten helper below exists
org_df.printSchema()

### Flatten helper
The helper below (built with Genie) flattens nested structs into `parent_child` columns; arrays are left intact. It keeps flattening until no struct columns remain.

In [0]:
# flatten the organizations' nested JSON structs into single-level columns
from pyspark.sql.types import StructType

def flatten_df(df):
    """Flatten nested structs into parent_child columns (arrays are left intact)."""
    while any(isinstance(df.schema[c].dataType, StructType) for c in df.columns):
        cols = []
        for c in df.columns:
            if isinstance(df.schema[c].dataType, StructType):
                for f in df.schema[c].dataType.fields:
                    cols.append(F.col(f"{c}.{f.name}").alias(f"{c}_{f.name}"))
            else:
                cols.append(F.col(c))
        df = df.select(*cols)
    return df

org_flat = flatten_df(org_df)
org_flat.printSchema()

In [0]:
# same load-time stamping as accounts; Ingestion_date is the partition key
org_flat = (org_flat.withColumn("Ingestion_timestamp", F.current_timestamp())
                        .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

### Write mode
Same as accounts: bronze is fully refreshed (overwrite) on every run. The commented variant below appends after the first load, for incremental ingestion.

In [0]:
# --- incremental write variant (DISABLED - kept for reference) ---
# appends when the bronze table already exists, overwrites only on the very
# first load. Switched off because the active cell below does a full refresh
# from the complete landing extracts on every run.

# org_table_name = "coreview.bronze.organizations"

# if spark.catalog.tableExists(org_table_name):
#     org_flat.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(org_table_name)
# else:
#     org_flat.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(org_table_name)


In [0]:
# full refresh: bronze always mirrors the current landing extract
org_table_name = "coreview.bronze.organizations"
org_flat.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(org_table_name)

In [0]:
# quick sanity check on the loaded row count
print(f"organizations: {org_flat.count()} rows")

In [0]:
# --- quick check of the loaded table (DISABLED) ---
# uncomment to preview the bronze table after the load
# %%sql
# select * from coreview.bronze.organizations

In [0]:
# landing volume location of the sign-in events extract (CSV files)
signins_path = "/Volumes/workspace/default/organizations_data/data/signins"

In [0]:
# read every CSV file in the folder as one DataFrame; columns stay raw
# strings - bronze keeps the values exactly as they landed
signins_df = spark.read.format('csv').option('header', 'true').load(signins_path)

In [0]:
# quick sanity check on the raw row count
print(f"signins: {signins_df.count()} rows")

In [0]:
# same load-time stamping as accounts; Ingestion_date is the partition key
signins_df = (signins_df.withColumn("Ingestion_timestamp", F.current_timestamp())
                        .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

In [0]:
# --- incremental write variant (DISABLED - kept for reference) ---
# appends when the bronze table already exists, overwrites only on the very
# first load. Switched off because the active cell below does a full refresh
# from the complete landing extracts on every run.

# signins_table_name = "coreview.bronze.signins"

# if spark.catalog.tableExists(signins_table_name):
#     signins_df.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(signins_table_name)
# else:
#     signins_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(signins_table_name)


In [0]:
# full refresh: bronze always mirrors the current landing extract
signins_table_name = "coreview.bronze.signins"
signins_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(signins_table_name)

In [0]:
# --- quick check of the loaded table (DISABLED) ---
# uncomment to preview the bronze table after the load
# %%sql
# select * from coreview.bronze.signins